In [1]:
import pandas as pd
import numpy as np
import re
import glob

In [2]:
df = pd.read_csv("../raw_data/amazon_india_2023.csv")

In [3]:
df.head()

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
0,TXN_2023_00000001,2023-01-16,CUST_2023_00005822,PROD_001899,Xiaomi Watch Premium,Electronics,Smart Watch,Xiaomi,59416.51,0.00,...,False,NaN,4/5,Returned,1,2023,1,0.06,True,4.2
1,TXN_2023_00000002,2023-01-15,CUST_2019_00019403,PROD_000472,Oppo R17 Pro 64GB Black,Electronics,Smartphones,Oppo,19474.76,0.00,...,False,NaN,4.0,Returned,1,2023,1,0.16,True,3.4
2,TXN_2023_00000003,2023-01-10,CUST_2023_00040022,PROD_001759,JBL Neckband,Electronics,Audio,JBL,27097.23,27.71,...,False,NaN,5.0,Delivered,1,2023,1,0.23,True,3.7
3,TXN_2023_00000004,2023-01-04,CUST_2023_00021818,PROD_001158,Xiaomi Redmi Note 12 256GB White,Electronics,Smartphones,Xiaomi,12883.85,0.00,...,False,NaN,3.5,Delivered,1,2023,1,0.23,True,4.1
4,TXN_2023_00000005,2023-01-07,CUST_2020_00048935,PROD_000221,Apple iPhone 8 32GB Blue,Electronics,Smartphones,Apple,165637.56,0.00,...,False,NaN,3.0,Delivered,1,2023,1,0.22,False,3.5


In [4]:
df["delivery_charges"].isna().sum(), len(df)

(np.int64(10162), 127132)

In [5]:
df["delivery_charges"].describe()

count    116970.000000
mean          0.000342
std           0.116956
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          40.000000
Name: delivery_charges, dtype: float64

In [6]:
df.shape

(127132, 34)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 127132 entries, 0 to 127131
Data columns (total 34 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   transaction_id          127132 non-null  object 
 1   order_date              127132 non-null  object 
 2   customer_id             127132 non-null  object 
 3   product_id              127132 non-null  object 
 4   product_name            127132 non-null  object 
 5   category                127132 non-null  object 
 6   subcategory             127132 non-null  object 
 7   brand                   127132 non-null  object 
 8   original_price_inr      127132 non-null  object 
 9   discount_percent        127132 non-null  float64
 10  discounted_price_inr    127132 non-null  float64
 11  quantity                127132 non-null  int64  
 12  subtotal_inr            127132 non-null  float64
 13  delivery_charges        116970 non-null  float64
 14  final_amount_inr    

In [8]:
df.columns

Index(['transaction_id', 'order_date', 'customer_id', 'product_id',
       'product_name', 'category', 'subcategory', 'brand',
       'original_price_inr', 'discount_percent', 'discounted_price_inr',
       'quantity', 'subtotal_inr', 'delivery_charges', 'final_amount_inr',
       'customer_city', 'customer_state', 'customer_tier',
       'customer_spending_tier', 'customer_age_group', 'payment_method',
       'delivery_days', 'delivery_type', 'is_prime_member', 'is_festival_sale',
       'festival_name', 'customer_rating', 'return_status', 'order_month',
       'order_year', 'order_quarter', 'product_weight_kg', 'is_prime_eligible',
       'product_rating'],
      dtype='object')

Question 1
Your dataset contains order_date in multiple formats: 'DD/MM/YYYY', 'DD-MM-YY', 'YYYY-MM-DD', and some invalid entries like '32/13/2020'. Clean and standardize all dates to 'YYYY-MM-DD' format, handling invalid dates appropriately.


In [9]:
df["order_date"].head(20)

0     2023-01-16
1     2023-01-15
2     2023-01-10
3     2023-01-04
4     2023-01-07
5     06-01-2023
6     2023-01-29
7     2023-01-08
8     2023-01-03
9     2023-01-05
10    2023-01-27
11    2023-01-08
12    2023-01-25
13    2023-01-26
14    2023-01-24
15    2023-01-06
16    2023-01-13
17    2023-01-09
18    2023-01-11
19    15/01/2023
Name: order_date, dtype: object

In [10]:
df["order_date"] = (
    df["order_date"]
    .str.replace(" ", "", regex=False)
    .str.replace("/", "-", regex=False)
)

parts = df["order_date"].str.split("-", expand=True)

year_last = parts[2].str.len() == 4

df.loc[year_last, "order_date"] = (
    parts[2] + "-" + parts[0] + "-" + parts[1]
)

parts = df["order_date"].str.split("-", expand=True)

mask = parts[1].astype(int) > 12

df.loc[mask, "order_date"] = (
    parts[0] + "-" + parts[2] + "-" + parts[1]
)

df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")

In [11]:
df["order_date"].min(), df["order_date"].max()

(Timestamp('2023-01-01 00:00:00'), Timestamp('2023-12-31 00:00:00'))

In [12]:
df["order_date"].isna().sum()

np.int64(0)

Question 2
The original_price_inr column contains mixed data types: numeric values, text with '₹' symbols, comma separators ('₹1,25,000'), and some entries like 'Price on Request'. Clean this column to contain only numeric values in Indian Rupees. 


In [13]:
df["original_price_inr"] = df["original_price_inr"].astype(str)

df["original_price_inr"] = df["original_price_inr"].str.replace("₹", "", regex=False)

df["original_price_inr"] = df["original_price_inr"].str.replace(",", "", regex=False)

df["original_price_inr"] = pd.to_numeric(df["original_price_inr"], errors="coerce")

In [14]:
df["original_price_inr"].unique()[:20]

array([ 59416.51,  19474.76,  27097.23,  12883.85, 165637.56, 102017.58,
        26405.22,  32445.27,  22067.75,       nan,  80242.05,  28593.41,
        53975.14,  40219.86,  25915.37,  17196.84,  78154.48, 123293.45,
        34831.05,  77327.51])

In [15]:
mask = df["original_price_inr"].isna()

df.loc[mask, "original_price_inr"] = np.where(
    df.loc[mask, "discount_percent"] == 0,
    
    # Case 1: no discount
    df.loc[mask, "discounted_price_inr"],
    
    # Case 2: discount present
    df.loc[mask, "discounted_price_inr"] / (1 - df.loc[mask, "discount_percent"] / 100)
)

In [16]:
df["original_price_inr"].dtypes

dtype('float64')

In [17]:
df["original_price_inr"].isna().sum()

np.int64(0)

Question 3
Customer ratings appear in various formats: '5.0', '4 stars', '3/5', '2.5/5.0', and some missing values. Standardize all ratings to numeric scale 1.0-5.0, handling inconsistent formats and missing values strategically.


In [18]:
df["customer_rating"] = df["customer_rating"].astype(str)

df["customer_rating"] = df["customer_rating"].str.replace(" stars", "", regex=False)

df["customer_rating"] = df["customer_rating"].str.split("/").str[0]

df["customer_rating"] = pd.to_numeric(df["customer_rating"], errors="coerce")

In [19]:
df["customer_rating"].describe()

count    88648.000000
mean         4.309212
std          0.573928
min          3.000000
25%          4.000000
50%          4.500000
75%          5.000000
max          5.000000
Name: customer_rating, dtype: float64

In [20]:
df["customer_rating"].value_counts().head(10)

customer_rating
4.5    29138
5.0    22747
4.0    22336
3.5     9044
3.0     5383
Name: count, dtype: int64

In [21]:
df["customer_rating"].isna().sum()

np.int64(38484)

In [22]:
df["customer_rating"].value_counts(dropna=False)

customer_rating
NaN    38484
4.5    29138
5.0    22747
4.0    22336
3.5     9044
3.0     5383
Name: count, dtype: int64

Question 4
The customer_city column has inconsistent naming: 'Bangalore/Bengaluru', 'Mumbai/Bombay', 'Delhi/New Delhi', along with spelling errors and case variations. Standardize all city names and handle geographical variations.


In [23]:
df["customer_city"] = df["customer_city"].str.strip().str.lower()

In [24]:
df["customer_city"].unique()

array(['kanpur', 'pune', 'lucknow', 'chennai', 'mumbai', 'surat',
       'visakhapatnam', 'chandigarh', 'bangalore', 'hyderabad', 'indore',
       'saharanpur', 'moradabad', 'gorakhpur', 'nagpur', 'coimbatore',
       'ahmedabad', 'allahabad', 'delhi', 'kochi', 'patna', 'kolkata',
       'jaipur', 'vadodara', 'meerut', 'varanasi', 'bhubaneswar',
       'bareilly', 'ludhiana', 'aligarh', 'calcutta', 'chenai',
       'delhi ncr', 'madras', 'bengaluru', 'bombay', 'mumba', 'bengalore',
       'new delhi', 'banglore'], dtype=object)

In [25]:
city_map = {
    "new delhi": "delhi",
    "delhi ncr": "delhi",

    "madras": "chennai",
    "chenai": "chennai",

    "calcutta": "kolkata",

    "bombay": "mumbai",
    "mumba": "mumbai",

    "bengalore": "bangalore",
    "banglore": "bangalore",
    "bengaluru": "bangalore"
}

In [26]:
df["customer_city"] = df["customer_city"].replace(city_map)

In [27]:
df["customer_city"] = df["customer_city"].str.title()

In [28]:
df["customer_city"].value_counts().head(20)

customer_city
Mumbai           14122
Delhi            12154
Bangalore         9856
Pune              8294
Chennai           8071
Kolkata           6521
Ahmedabad         6418
Jaipur            5272
Surat             5094
Nagpur            4696
Hyderabad         4319
Lucknow           4203
Kanpur            4175
Indore            4142
Coimbatore        3396
Kochi             3154
Visakhapatnam     2832
Patna             2723
Vadodara          2689
Bhubaneswar       2618
Name: count, dtype: int64

In [29]:
df["customer_city"].unique()

array(['Kanpur', 'Pune', 'Lucknow', 'Chennai', 'Mumbai', 'Surat',
       'Visakhapatnam', 'Chandigarh', 'Bangalore', 'Hyderabad', 'Indore',
       'Saharanpur', 'Moradabad', 'Gorakhpur', 'Nagpur', 'Coimbatore',
       'Ahmedabad', 'Allahabad', 'Delhi', 'Kochi', 'Patna', 'Kolkata',
       'Jaipur', 'Vadodara', 'Meerut', 'Varanasi', 'Bhubaneswar',
       'Bareilly', 'Ludhiana', 'Aligarh'], dtype=object)

Question 5
Boolean columns (is_prime_member, is_prime_eligible, is_festival_sale) contain mixed values: True/False, Yes/No, 1/0, Y/N, and some missing entries. Convert all boolean columns to consistent True/False format.


In [30]:
bool_candidates = []

bool_values = {"true","false","yes","no","y","n","1","0"}

for col in df.columns:
    vals = set(df[col].astype(str).str.lower().dropna().unique())
    
    if vals & bool_values:
        bool_candidates.append(col)

bool_candidates

['quantity',
 'delivery_days',
 'is_prime_member',
 'is_festival_sale',
 'order_month',
 'order_quarter',
 'is_prime_eligible']

In [31]:
boolean_cols = ["is_prime_member", "is_prime_eligible", "is_festival_sale"]

bool_map = {
    "true": True,
    "false": False,
    "yes": True,
    "no": False,
    "y": True,
    "n": False,
    "1": True,
    "0": False
}

for col in boolean_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(bool_map)
    )

In [32]:
df[boolean_cols].value_counts(dropna=False)

is_prime_member  is_prime_eligible  is_festival_sale
True             True               False               40792
False            True               False               32129
True             True               True                18048
False            True               True                14094
True             False              False                8685
False            False              False                6633
True             False              True                 3867
False            False              True                 2884
Name: count, dtype: int64

Question 6
Product categories have variations: 'Electronics/Electronic/ELECTRONICS/Electronics & Accessories'. Standardize category names across the dataset and ensure consistent naming conventions.


In [3]:
df["category"].value_counts().head(20)

category
Electronics                  127052
Electronics & Accessories        26
Electronicss                     25
Electronic                       17
ELECTRONICS                      12
Name: count, dtype: int64

In [7]:
category_map = {
    "ELECTRONICS": "Electronics",
    "Electronics & Accessories": "Electronics",
    "Electronic": "Electronics",
    "Electronicss": "Electronics"
}

In [8]:
df["category"] = df["category"].replace(category_map)
df["category"] = df["category"].str.title()

In [9]:
df["category"].value_counts()

category
Electronics    127132
Name: count, dtype: int64

Question 7
The delivery_days column contains negative values, text entries like 'Same Day', '1-2 days', and some unrealistic values like 50 days. Clean this column to contain only valid numeric delivery days.


In [37]:
df["delivery_days"].unique()

array(['3', '4', '1', '2', '5', '6', 'Same Day', '-1', '7', '15',
       '1-2 days', 'Express', '0'], dtype=object)

In [38]:
df["delivery_days"] = df["delivery_days"].astype(str).str.strip().str.lower()

In [39]:
df["delivery_days"] = df["delivery_days"].replace({
    "same day": "0",
    "express": "1"
})

In [40]:
df["delivery_days"] = df["delivery_days"].str.extract(r"(-?\d+)")

In [41]:
df["delivery_days"] = pd.to_numeric(df["delivery_days"], errors="coerce")

In [42]:
df.loc[df["delivery_days"] < 0, "delivery_days"] = None

In [43]:
df["delivery_days"].unique()

array([ 3.,  4.,  1.,  2.,  5.,  6.,  0., nan,  7., 15.])

In [44]:
df["delivery_days"].isnull().sum()

np.int64(755)

In [45]:
df.loc[df["delivery_days"] < 0, "delivery_days"] = np.nan

df["delivery_days"] = df["delivery_days"].fillna(df["delivery_days"].median())

In [46]:
df["delivery_days"].describe()

count    127132.000000
mean          2.863355
std           1.743126
min           0.000000
25%           1.000000
50%           3.000000
75%           4.000000
max          15.000000
Name: delivery_days, dtype: float64

In [47]:
df["delivery_days"].isna().sum()

np.int64(0)

Question 8
Identify and handle duplicate transactions where the same customer, product, date, and amount appear multiple times. Some duplicates are genuine (bulk orders) while others are data errors. Develop a strategy to distinguish and handle both cases.


In [48]:
duplicate_mask = df.duplicated(
    subset=["customer_id", "product_id", "order_date", "original_price_inr"],
    keep=False
)

duplicates = df[duplicate_mask]

In [49]:
duplicates.head()

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
251,TXN_2023_00000252,2023-01-31,CUST_2023_00028413,PROD_000668,Samsung Galaxy S20 128GB White,Electronics,Smartphones,Samsung,91130.98,0.00,...,False,NaN,NaN,Delivered,1,2023,1,0.21,True,3.2
252,TXN_2023_00000253,2023-01-29,CUST_2023_00037111,PROD_000171,Xiaomi Redmi Note 3 16GB Black,Electronics,Smartphones,Xiaomi,28581.43,0.00,...,False,NaN,4.5,Delivered,1,2023,1,0.20,False,4.3
326,TXN_2023_00000327,2023-01-14,CUST_2022_00008910,PROD_000885,Xiaomi Poco F3 256GB White,Electronics,Smartphones,Xiaomi,19739.71,0.00,...,False,NaN,NaN,Delivered,1,2023,1,0.23,True,4.4
337,TXN_2023_00000338,2023-01-09,CUST_2023_00026453,PROD_000592,Realme Realme X 128GB Blue,Electronics,Smartphones,Realme,19456.03,0.00,...,False,NaN,3.5,Delivered,1,2023,1,0.16,False,4.4
345,TXN_2023_00000346,2023-01-07,CUST_2023_00010735,PROD_001105,Samsung Galaxy S23 256GB Black,Electronics,Smartphones,Samsung,133706.91,19.43,...,False,NaN,4.5,Cancelled,1,2023,1,0.18,True,3.2


In [50]:
duplicate_mask = df.duplicated(
    subset=["customer_id", "product_id", "order_date", "original_price_inr"],
    keep=False
)

duplicates = df[duplicate_mask]

In [51]:
duplicates.shape

(1240, 34)

In [52]:
df.duplicated().sum()

np.int64(0)

In [53]:
duplicates.sort_values(
    ["customer_id","product_id","order_date"]
).head(10)

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
117708,TXN_2023_00117709,2023-12-08,CUST_2015_00000167,PROD_000974,Samsung Galaxy S22 64GB Blue,Electronics,Smartphones,Samsung,83909.01,0.00,...,False,NaN,NaN,Returned,12,2023,4,0.21,True,4.0
126946,TXN_2023_00117709_DUP,2023-12-08,CUST_2015_00000167,PROD_000974,Samsung Galaxy S22 64GB Blue,Electronics,Smartphones,Samsung,83909.01,0.00,...,False,NaN,NaN,Returned,12,2023,4,0.21,True,4.0
62618,TXN_2023_00062619,2023-07-19,CUST_2015_00001651,PROD_000400,Xiaomi Poco F1 128GB White,Electronics,Smartphones,Xiaomi,28832.10,0.00,...,False,NaN,4.0,Delivered,7,2023,3,0.16,False,4.0
126535,TXN_2023_00062619_DUP,2023-07-19,CUST_2015_00001651,PROD_000400,Xiaomi Poco F1 128GB White,Electronics,Smartphones,Xiaomi,28832.10,0.00,...,False,NaN,4.0,Delivered,7,2023,3,0.16,False,4.0
106001,TXN_2023_00106002,2023-11-26,CUST_2015_00002377,PROD_001882,Apple Sports Watch Premium,Electronics,Smart Watch,Apple,5675.91,0.00,...,False,NaN,5.0,Delivered,11,2023,4,0.07,True,4.0
126763,TXN_2023_00106002_DUP,2023-11-26,CUST_2015_00002377,PROD_001882,Apple Sports Watch Premium,Electronics,Smart Watch,Apple,5675.91,0.00,...,False,NaN,5.0,Delivered,11,2023,4,0.07,True,4.0
65174,TXN_2023_00065175,2023-07-21,CUST_2015_00005849,PROD_001538,Dell Inspiron 4GB RAM Black,Electronics,Laptops,Dell,41161.52,0.00,...,False,NaN,NaN,Delivered,7,2023,3,2.37,True,4.0
126675,TXN_2023_00065175_DUP,2023-07-21,CUST_2015_00005849,PROD_001538,Dell Inspiron 4GB RAM Black,Electronics,Laptops,Dell,41161.52,0.00,...,False,NaN,NaN,Delivered,7,2023,3,2.37,True,4.0
62374,TXN_2023_00062375,2023-07-16,CUST_2015_00006277,PROD_001009,OnePlus OnePlus Nord CE 2 64GB Blue,Electronics,Smartphones,OnePlus,78981.32,25.74,...,True,Prime Day,NaN,Delivered,7,2023,3,0.17,True,3.8
126700,TXN_2023_00062375_DUP,2023-07-16,CUST_2015_00006277,PROD_001009,OnePlus OnePlus Nord CE 2 64GB Blue,Electronics,Smartphones,OnePlus,78981.32,25.74,...,True,Prime Day,NaN,Delivered,7,2023,3,0.17,True,3.8


In [54]:
duplicates.groupby(
    ["customer_id","product_id","order_date","original_price_inr"]
).size().sort_values(ascending=False).head(10)

customer_id         product_id   order_date  original_price_inr
CUST_2015_00000167  PROD_000974  2023-12-08  83909.01              2
CUST_2015_00001651  PROD_000400  2023-07-19  28832.10              2
CUST_2015_00002377  PROD_001882  2023-11-26  5675.91               2
CUST_2015_00005849  PROD_001538  2023-07-21  41161.52              2
CUST_2015_00006277  PROD_001009  2023-07-16  78981.32              2
CUST_2015_00006522  PROD_001904  2023-12-12  50383.41              2
CUST_2015_00007131  PROD_000808  2023-08-22  241958.43             2
CUST_2015_00008514  PROD_000415  2023-11-02  34218.11              2
CUST_2015_00009846  PROD_000107  2023-01-30  186974.26             2
CUST_2015_00009916  PROD_000029  2023-03-04  71884.95              2
dtype: int64

In [55]:
dup_groups = df.groupby(
    ["customer_id","product_id","order_date","original_price_inr"]
).size().reset_index(name="count")

dup_groups = dup_groups[dup_groups["count"] > 1]

In [56]:
dup_rows = df.merge(
    dup_groups,
    on=["customer_id","product_id","order_date","original_price_inr"],
    how="inner"
)

In [57]:
dup_rows[["customer_id","product_id","quantity","count"]].head()

,customer_id,product_id,quantity,count
0,CUST_2023_00028413,PROD_000668,1,2
1,CUST_2023_00037111,PROD_000171,1,2
2,CUST_2022_00008910,PROD_000885,1,2
3,CUST_2023_00026453,PROD_000592,1,2
4,CUST_2023_00010735,PROD_001105,1,2


In [58]:
df_clean = df.drop_duplicates(
    subset=["customer_id","product_id","order_date","original_price_inr"],
    keep="first"
)

In [59]:
df_clean.duplicated(
    subset=["customer_id","product_id","order_date","original_price_inr"]
).sum()

np.int64(0)

In [60]:
df["transaction_id"].duplicated().sum()

np.int64(0)

In [61]:
df = df.drop_duplicates(subset="transaction_id", keep="first")

In [62]:
df[df["transaction_id"]=="TXN_2015_00000280"]

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating


Question 9
The dataset contains outlier prices where some products show prices 100x higher than expected due to data entry errors (decimal point issues). Identify and correct these outliers using statistical methods and domain knowledge.


In [63]:

# ── FIX 1: Negative prices ──────────────────────────
neg_mask = df["original_price_inr"] < 0
df.loc[neg_mask, "original_price_inr"] = df.loc[neg_mask, "original_price_inr"].abs()
print(f"Negative prices fixed: {neg_mask.sum()}")

# ── FIX 2: Outliers ──────────────────────────────────
subcategory_caps = {
    "Smart Watch":        100000,
    "Tablets":            180000,
    "Smartphones":        400000,
    "Laptops":            550000,
    "TV & Entertainment": 500000,
    "Audio":              200000,
}

outlier_mask = df.apply(
    lambda row: row["original_price_inr"] > subcategory_caps.get(row["subcategory"], 999999),
    axis=1
)
df.loc[outlier_mask, "original_price_inr"] = (
    df.loc[outlier_mask, "original_price_inr"] / 100
).round(2)
print(f"Outliers fixed: {outlier_mask.sum()}")

# ── FIX 3: delivery_charges ──────────────────────────
df["delivery_charges"] = df["delivery_charges"].fillna(0)

# ── FIX 4: Recalculate ───────────────────────────────
df["discounted_price_inr"] = (df["original_price_inr"] * (1 - df["discount_percent"] / 100)).round(2)
df["subtotal_inr"] = (df["discounted_price_inr"] * df["quantity"]).round(2)
df["final_amount_inr"] = (df["subtotal_inr"] + df["delivery_charges"]).round(2)

# ── VERIFY ───────────────────────────────────────────
print(df.groupby("subcategory", observed=True)["original_price_inr"]
      .describe()[["min","max","mean","50%"]].round(2))
print(f"\nNaN in final_amount_inr:   {df['final_amount_inr'].isna().sum()}")
print(f"Negative prices remaining: {(df['original_price_inr'] < 0).sum()}")


Negative prices fixed: 326
Outliers fixed: 450
                         min        max      mean        50%
subcategory                                                 
Audio                1076.52  195515.50  21107.28   22289.13
Laptops              6128.28  479121.90  95551.75   80560.33
Smart Watch          2046.11   72415.58  40319.39   39824.99
Smartphones          4096.62  396154.90  57181.55   35945.58
TV & Entertainment  12567.14  431283.40  96782.44  100304.51
Tablets              2833.34  168533.24  79442.89   78053.94

NaN in final_amount_inr:   0
Negative prices remaining: 0


In [64]:
# Check if any legitimate products were over-corrected in cleaned files
for sub, cap in subcategory_caps.items():
    over = df[(df["subcategory"] == sub) & 
                    (df["original_price_inr"] > cap)]
    if len(over) > 0:
        print(f"\n{sub} (cap ₹{cap:,}): {len(over)} rows over")
        print(over[["product_name", "original_price_inr"]].drop_duplicates().head(5))
    else:
        print(f"\n{sub}: ✅ All within cap")


Smart Watch: ✅ All within cap

Tablets: ✅ All within cap

Smartphones: ✅ All within cap

Laptops: ✅ All within cap

TV & Entertainment: ✅ All within cap

Audio: ✅ All within cap


Question 10
Payment methods contain inconsistent naming: 'UPI/PhonePe/GooglePay', 'Credit Card/CREDIT_CARD/CC', 
'Cash on Delivery/COD/C.O.D'. Standardize payment method categories and create a clean categorical hierarchy.


In [65]:
# ── Standardize payment methods ────────────────────
payment_standardize = {
    # UPI variants
    "UPI": "UPI", "PhonePe": "UPI", "GooglePay": "UPI", "Google Pay": "UPI",
    "UPI/PhonePe": "UPI", "UPI/GooglePay": "UPI",

    # Credit Card variants
    "Credit Card": "Credit Card", "CREDIT_CARD": "Credit Card", "CC": "Credit Card",

    # Debit Card variants
    "Debit Card": "Debit Card", "DEBIT_CARD": "Debit Card", "DC": "Debit Card",

    # COD variants
    "Cash on Delivery": "COD", "COD": "COD", "C.O.D": "COD",

    # Others
    "Wallet": "Wallet",
    "Net Banking": "Net Banking",
    "BNPL": "BNPL"
}

df["payment_method"] = df["payment_method"].map(payment_standardize).fillna(df["payment_method"])

# ── Create categorical hierarchy ───────────────────
payment_category = {
    "UPI":          "Digital Payment",
    "Wallet":       "Digital Payment",
    "Net Banking":  "Digital Payment",
    "Credit Card":  "Card Payment",
    "Debit Card":   "Card Payment",
    "BNPL":         "Pay Later",
    "COD":          "Cash on Delivery"
}

df["payment_category"] = df["payment_method"].map(payment_category).astype("category")

# ── Verify ─────────────────────────────────────────
print(df["payment_method"].value_counts())
print(f"\nNaN in payment_category: {df['payment_category'].isna().sum()}")

payment_method
UPI            66192
Credit Card    17653
COD            15366
Debit Card     12561
Net Banking     6386
BNPL            5090
Wallet          3884
Name: count, dtype: int64

NaN in payment_category: 0


Handling Nan - in customer age group

In [66]:
df["customer_age_group"] = df["customer_age_group"].fillna("Unknown")

Checking for object columns

In [67]:
df.select_dtypes(include="object").columns

Index(['transaction_id', 'customer_id', 'product_id', 'product_name',
       'category', 'subcategory', 'brand', 'customer_city', 'customer_state',
       'customer_tier', 'customer_spending_tier', 'customer_age_group',
       'payment_method', 'delivery_type', 'festival_name', 'return_status'],
      dtype='object')

In [68]:
# ── Optimize memory: convert to category dtype ───────
cat_columns = ["category", "subcategory", "customer_tier", 
               "customer_spending_tier", "customer_age_group",
               "payment_method", "delivery_type", 
               "festival_name", "return_status"]

df[cat_columns] = df[cat_columns].astype("category")

# Verify
print(df[cat_columns].dtypes)
print(f"\nMemory usage after optimization:")
print(df.memory_usage(deep=True).sum() / 1024**2, "MB")

category                  category
subcategory               category
customer_tier             category
customer_spending_tier    category
customer_age_group        category
payment_method            category
delivery_type             category
festival_name             category
return_status             category
dtype: object

Memory usage after optimization:
69.27239418029785 MB


In [69]:
# NaN summary for all columns
nan_summary = df.isna().sum()
nan_summary = nan_summary[nan_summary > 0].sort_values(ascending=False)

print(f"Total columns with NaN: {len(nan_summary)}")
print(f"Total rows in dataset: {df.shape[0]}")
print(f"\nNaN counts and percentages:")
print(pd.DataFrame({
    "NaN Count": nan_summary,
    "Percentage": (nan_summary / df.shape[0] * 100).round(2)
}))

Total columns with NaN: 2
Total rows in dataset: 127132

NaN counts and percentages:
                 NaN Count  Percentage
festival_name        88239       69.41
customer_rating      38484       30.27


In [70]:
df.to_csv("data_cleaning_2023.csv", index=False)
print("File saved successfully!")

File saved successfully!
